# BGE-M3 Baseline Retrieval
## All-fields dense retrieval baseline over 98,716 companies

**Why BGE-M3?** Unlike `BAAI/bge-large-en-v1.5` (tested in notebooks 02/13/14), BGE-M3 is multilingual (100+ languages) and was specifically trained so that **no instruction prefix is needed on either the query or document side** for retrieval -- confirmed via the official model card. This matters because our corpus has genuine non-English company text. 1024-dim dense embeddings, cosine similarity.

**Text representation:** all fields -- name, country, state, city, district, organization_type, organization_size, nace_code, summary, summary_keywords (identical convention to `03_baseline_minilm.ipynb`, for a fair comparison).

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

RESULT_DIR = Path('result/16_baseline_bgem3')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[Setup] Result folder : {RESULT_DIR}/ -- ready')

In [ ]:
import json, time
import numpy as np
import pandas as pd
import faiss
print('[Imports] All packages loaded successfully')

## 1. Load dataset & build all-fields rich text

In [ ]:
print('[Load] Loading production results...')
results_df = pd.read_excel('dataset/production_results.xlsx')
all_companies = results_df.drop_duplicates(subset='domain').reset_index(drop=True)
print(f'[Load] Unique companies : {len(all_companies):,}')

with open('dataset/goi_search_results.json', 'r') as f:
    queries_data = json.load(f)
print(f'[Load] Queries : {len(queries_data)}')

def build_rich_text(row):
    """Combine all informative fields into one string -- same convention as 03_baseline_minilm.ipynb."""
    parts = []
    for field, prefix in [
        ('name',              'Company:'),
        ('country',           'Country:'),
        ('state',             'State:'),
        ('municipality',      'City:'),
        ('district',          'District:'),
        ('organization_type', 'Type:'),
        ('organization_size', 'Size:'),
        ('nace_code',         'Industry:'),
        ('summary',           ''),
    ]:
        val = row.get(field, '')
        if isinstance(val, str) and val.strip():
            parts.append(f'{prefix} {val}'.strip() if prefix else val)
    kw = row.get('summary_keywords', '')
    if isinstance(kw, str) and kw.strip():
        kw_clean = kw.replace("'", '').replace('[', '').replace(']', '')
        parts.append(f'Keywords: {kw_clean}')
    return ' | '.join(parts)

print('[Load] Building all-fields rich text for each company...')
rich_texts = [build_rich_text(row) for _, row in all_companies.iterrows()]
print('[Load] Sample rich text (first company):')
print(f'  {rich_texts[0][:300]}...')

## 2. GPU check

In [ ]:
import torch

print(f'[GPU] CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[GPU] Device         : {torch.cuda.get_device_name(0)}')
    print(f'[GPU] VRAM           : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('[GPU] WARNING: No GPU -- this model needs a GPU to be practical')
    DEVICE = 'cpu'

## 3. Encode all companies with BGE-M3 (no instruction prefix needed)

In [ ]:
from sentence_transformers import SentenceTransformer

print('[Encode] Loading BGE-M3...')
t0    = time.time()
model = SentenceTransformer('BAAI/bge-m3', device=DEVICE)
print(f'[Encode] Model loaded in {time.time()-t0:.1f}s on {model.device}')

print('[Encode] Encoding all companies (no instruction prefix -- BGE-M3 does not need one)...')
t0 = time.time()
embeddings = model.encode(
    rich_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
ENCODE_TIME = time.time() - t0
print(f'[Encode] Done in {ENCODE_TIME/60:.1f} minutes')
print(f'[Encode] Embeddings shape : {embeddings.shape}')
np.save(RESULT_DIR / 'company_embeddings.npy', embeddings)

## 4. Build FAISS index (IndexFlatIP, cosine via normalized vectors)

In [ ]:
embeddings = np.load(RESULT_DIR / 'company_embeddings.npy').astype('float32')
dimension  = embeddings.shape[1]
index      = faiss.IndexFlatIP(dimension)
index.add(embeddings)
faiss.write_index(index, str(RESULT_DIR / 'company_faiss.index'))
print(f'[FAISS] Index built, {index.ntotal:,} vectors, dim={dimension}')

## 5. Run all 101 queries (no instruction prefix)

In [ ]:
print(f'[Run] Starting BGE-M3 retrieval for {len(queries_data)} queries...')
all_results = []
query_times = []
total_start = time.time()

for i, item in enumerate(queries_data):
    query_id = item['query_id']
    query    = item['query']

    t0        = time.perf_counter()
    query_emb = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    scores, idxs = index.search(query_emb, 1000)
    query_ms  = (time.perf_counter() - t0) * 1000
    query_times.append(query_ms)

    for rank, (idx, score) in enumerate(zip(idxs[0], scores[0])):
        company = all_companies.iloc[idx]
        all_results.append({
            'query_id': query_id, 'query': query, 'rank': rank + 1,
            'score': float(score), 'domain': company['domain'],
            'name': company.get('name', ''), 'country': company.get('country', ''),
        })

    if (i + 1) % 20 == 0 or (i + 1) == len(queries_data):
        elapsed = time.time() - total_start
        print(f'[Run] {i+1:3d}/{len(queries_data)}  |  avg {sum(query_times)/len(query_times):.1f}ms/query')

results_df_out = pd.DataFrame(all_results)
results_df_out.to_csv(RESULT_DIR / 'bgem3_results.csv', index=False)
AVG_LATENCY_MS = sum(query_times) / len(query_times)
print(f'[Run] Done! Avg latency: {AVG_LATENCY_MS:.1f}ms')

## Evaluation -- NDCG, Precision, Recall, F1 @ k (same protocol as all other baselines)

In [ ]:
print('[Eval] Loading production labels...')
production_df = pd.read_excel('dataset/production_results.xlsx')

K_VALUES = [10, 50, 100, 500, 1000]

def get_relevant(query_id, top_k=1000):
    return set(production_df[
        (production_df['query_id'] == query_id) &
        (production_df['rank'] <= top_k)
    ]['domain'].tolist())

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k if k else 0

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0

def f1_at_k(retrieved, relevant, k):
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0

def dcg_at_k(retrieved, relevant, k):
    return sum(
        1 / np.log2(i + 2)
        for i, d in enumerate(retrieved[:k]) if d in relevant
    )

def ndcg_at_k(retrieved, relevant, k):
    ideal = dcg_at_k(list(relevant), relevant, k)
    return dcg_at_k(retrieved, relevant, k) / ideal if ideal else 0

print('[Eval] Computing metrics for all queries...')
eval_rows = []

for i, item in enumerate(queries_data):
    qid       = item['query_id']
    query     = item['query']
    relevant  = get_relevant(qid)
    retrieved = (
        results_df_out[results_df_out['query_id'] == qid]
        .sort_values('rank')['domain'].tolist()
    )
    for k in K_VALUES:
        eval_rows.append({
            'query_id':  qid,
            'query':     query,
            'k':         k,
            'precision': precision_at_k(retrieved, relevant, k),
            'recall':    recall_at_k(retrieved, relevant, k),
            'f1':        f1_at_k(retrieved, relevant, k),
            'ndcg':      ndcg_at_k(retrieved, relevant, k),
        })

    if (i + 1) % 25 == 0:
        print(f'[Eval] {i+1}/101 queries evaluated...')

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(RESULT_DIR / 'evaluation_bgem3.csv', index=False)
print(f'[Eval] Saved to {RESULT_DIR}/evaluation_bgem3.csv')

## Final Summary

In [ ]:
print('[Summary] ============================================================')
print('[Summary] BGE-M3 RESULTS (All-Fields)')
print(f'\n[Summary] Encoding time     : {ENCODE_TIME/60:.1f} minutes')
print(f'[Summary] Avg query latency : {AVG_LATENCY_MS:.1f}ms')
print(f'[Summary] Companies encoded : {len(all_companies):,}')
print('[Summary] Embedding dims    : 1024')
print(f'[Summary] Queries evaluated : {len(queries_data)}')
print()
print(f'  {"k":<6} {"NDCG":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('  ' + '-' * 46)
for k in K_VALUES:
    sub = eval_df[eval_df['k'] == k]
    print(f'  {k:<6} '
          f'{sub["ndcg"].mean():>8.3f} '
          f'{sub["precision"].mean():>10.3f} '
          f'{sub["recall"].mean():>8.3f} '
          f'{sub["f1"].mean():>8.3f}')

print()
print('[Summary] Comparison against models already tested (Recall@1000):')
print('  BM25              : 0.635')
print('  Nomic             : 0.684')
print('  BGE summary-only  : 0.707  (prefix-fixed)')
print('  BGE all-fields    : 0.727  (prefix-fixed)')
print('  OpenAI large      : 0.728')
print('  MiniLM            : 0.741  (best so far)')
r1000 = eval_df[eval_df["k"] == 1000]["recall"].mean()
print(f'  BGE-M3            : {r1000:.3f}')